<a href="https://colab.research.google.com/github/tkoziara/Piper-Tools/blob/main/Training_EN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0) Verify Colab GPU and environment

In [ ]:
import torch, sys
print('Python', sys.version.split()[0])
print('PyTorch', torch.__version__)
print('CUDA available', torch.cuda.is_available())
# show GPU info if available
if torch.cuda.is_available():
    try:
        import subprocess
        print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode().strip())
    except Exception as e:
        print('nvidia-smi failed:', e)

## 1) Bootstrap environment (run once)

In [ ]:
%%bash
set -e
echo 'Colab bootstrap: installing build deps and piper1-gpl (editable)'
python3 -m pip install --upgrade pip setuptools wheel
python3 -m pip install --upgrade scikit-build cmake ninja wheel setuptools
if [ ! -d /content/piper1-gpl ]; then
  if git ls-remote git@github.com:tkoziara/piper1-gpl.git >/dev/null 2>&1; then
    git clone git@github.com:tkoziara/piper1-gpl.git /content/piper1-gpl
  else
    git clone https://github.com/tkoziara/piper1-gpl.git /content/piper1-gpl
  fi
fi
cd /content/piper1-gpl
python3 -m pip install -e ".[train]" || true
bash ./build_monotonic_align.sh || true
python3 setup.py build_ext --inplace -v || true
python3 -m pip install --upgrade onnxscript flask openai-whisper soundfile librosa lightning pytorch-lightning pysilero-vad pathvalidate jsonargparse[signatures] || true
python3 - <<'PY'
import torch, sys
print('Python', sys.version.split()[0])
print('PyTorch', torch.__version__)
print('CUDA available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device', torch.cuda.get_device_name(0))
PY

## 2.1) Mount Drive and set paths/hyperparameters


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
VOICE_NAME = 'tomek_en'
# Maximum number of training epochs (set to None to leave Trainer default)
MAX_EPOCHS = 100
ESPEAK_VOICE = 'en-us'
# Shuffle settings: strong = full per-epoch shuffle + worker RNG, normal = shuffle=True,
# weak = no DataLoader shuffle but otherwise normal batching, off = deterministic order.
SHUFFLE = 'strong'
SAMPLE_RATE = 22050
# Primary batch size used by the notebook; training launcher will use this as an override if set
BATCH_SIZE = 8
# Learning rate settings (manual defaults)
LEARNING_RATE = 2e-4
LEARNING_RATE_D = 1e-4
LR_DECAY = 0.999875
LR_DECAY_D = 0.9999
# Optional training phase presets: set TRAINING_PHASE = 'alignment' etc. and run cell 2.2
TRAINING_PHASE = None
# Optional overrides for Colab/T4 memory tuning (can be adjusted here)
NUM_WORKERS = 2
SEGMENT_SIZE = 6144
PRECISION = '16'
# Runtime Drive sync: set ENABLE_CHECKPOINT_MONITOR=True to automatically copy new checkpoints
# to Drive while training runs. CHECKPOINT_POLL_INTERVAL controls how often (in seconds) the
# background thread wakes up to check for and copy a new checkpoint to Drive as VOICE_NAME-latest.ckpt.
ENABLE_CHECKPOINT_MONITOR = True
CHECKPOINT_POLL_INTERVAL = 300
DRIVE_ROOT = Path('/content/drive/MyDrive/tts_data')
DATA_ROOT = DRIVE_ROOT / VOICE_NAME
AUDIO_DIR = DATA_ROOT / 'wavs'
CSV_PATH = DATA_ROOT / 'metadata.csv'
CACHE_DIR = Path('/content/piper_cache')
CKPT_PATH = DRIVE_ROOT / f"{VOICE_NAME}-latest.ckpt"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('CSV exists', CSV_PATH.exists())
print('Audio dir exists', AUDIO_DIR.exists())
print('Checkpoint exists', CKPT_PATH.exists())


## 2.2) Optional training phase presets


In [ ]:
# Set TRAINING_PHASE to one of: warmup, alignment, core_training, consolidation, fine_tuning, all
# Example: TRAINING_PHASE = "alignment"
PHASE_ORDER = ["warmup", "alignment", "core_training", "consolidation", "fine_tuning"]
PHASE_CONFIGS = {
    "warmup": {
        "MAX_EPOCHS": 10,
        "BATCH_SIZE": 8,
        "LEARNING_RATE": 5e-5,
        "LEARNING_RATE_D": 2.0e-5,
        "LR_DECAY": 0.9999,
        "LR_DECAY_D": 0.9999,
    },
    "alignment": {
        "MAX_EPOCHS": 20,
        "BATCH_SIZE": 8,
        "LEARNING_RATE": 7e-5,
        "LEARNING_RATE_D": 2.8e-5,
        "LR_DECAY": 0.9999,
        "LR_DECAY_D": 0.9999,
    },
    "core_training": {
        "MAX_EPOCHS": 90,
        "BATCH_SIZE": 12,
        "LEARNING_RATE": 9e-5,
        "LEARNING_RATE_D": 3.5e-5,
        "LR_DECAY": 0.9999,
        "LR_DECAY_D": 0.9999,
    },
    "consolidation": {
        "MAX_EPOCHS": 80,
        "BATCH_SIZE": 18,
        "LEARNING_RATE": 6e-5,
        "LEARNING_RATE_D": 2.2e-5,
        "LR_DECAY": 0.99993,
        "LR_DECAY_D": 0.99993,
    },
    "fine_tuning": {
        "MAX_EPOCHS": 60,
        "BATCH_SIZE": 22,
        "LEARNING_RATE": 4e-5,
        "LEARNING_RATE_D": 1.5e-5,
        "LR_DECAY": 0.99994,
        "LR_DECAY_D": 0.99994,
    },
}

if TRAINING_PHASE is not None and TRAINING_PHASE != "all":
    if TRAINING_PHASE not in PHASE_CONFIGS:
        raise ValueError(
            f"Unknown TRAINING_PHASE: {TRAINING_PHASE!r}. Valid values: {list(PHASE_CONFIGS) + ['all']}"
        )
    phase = PHASE_CONFIGS[TRAINING_PHASE]
    print("Applying training phase preset:", TRAINING_PHASE)
    for key, value in phase.items():
        globals()[key] = value

print("TRAINING_PHASE =", TRAINING_PHASE)
print("MAX_EPOCHS =", MAX_EPOCHS)
print("BATCH_SIZE =", BATCH_SIZE)
print("LEARNING_RATE =", LEARNING_RATE)
print("LEARNING_RATE_D =", LEARNING_RATE_D)
print("LR_DECAY =", LR_DECAY)
print("LR_DECAY_D =", LR_DECAY_D)


## 3) Sanitize checkpoint and run training

In [ ]:
# Cell 3 — checkpoint migration + training launcher
import sys, os, torch, pathlib, tempfile, subprocess, textwrap
from pathlib import Path

os.environ.setdefault("PYTORCH_ALLOC_CONF", "max_split_size_mb:128,allocator:default")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128")
torch.backends.cudnn.benchmark = False

PIPER_SRC = Path("/content/piper1-gpl/src")
if PIPER_SRC.exists():
    sys.path.insert(0, str(PIPER_SRC))

print("DEBUG: VOICE_NAME =", globals().get("VOICE_NAME"))
print("DEBUG: DATA_ROOT  =", globals().get("DATA_ROOT"))
print("DEBUG: CKPT_PATH  =", globals().get("CKPT_PATH"))

DATA_ROOT = Path(DATA_ROOT)
CONFIG_PATH = DATA_ROOT / f"{VOICE_NAME}.json"
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Missing required config: {CONFIG_PATH}")

CKPT_PATH = Path(CKPT_PATH) if "CKPT_PATH" in globals() and CKPT_PATH is not None else None

def _get_pl_version():
    try:
        import lightning.pytorch as _lp
        return getattr(_lp, "__version__", None) or getattr(_lp, "version", None)
    except Exception:
        try:
            import lightning as _l
            return getattr(_l, "__version__", None) or getattr(_l, "version", None)
        except Exception:
            return "unknown"

pl_version = _get_pl_version()

def run_training_phase(phase_name=None):
    if phase_name is not None:
        print(f"=== RUNNING TRAINING PHASE: {phase_name} ===")

    use_ckpt = None
    tmp_ckpt = None
    if CKPT_PATH and CKPT_PATH.exists():
        print("DEBUG: loading checkpoint for weights extraction:", CKPT_PATH)
        ckpt = torch.load(str(CKPT_PATH), map_location="cpu", weights_only=False)
        ckpt.pop("hyper_parameters", None)
        ckpt.pop("hparams", None)
        minimal = {k: v for k, v in ckpt.items() if k in ("state_dict", "optimizer_states", "lr_schedulers", "epoch", "global_step")}
        if not minimal:
            minimal = ckpt
        minimal.setdefault("pytorch-lightning_version", pl_version)
        minimal.setdefault("pytorch_lightning_version", pl_version)
        fd, tmp_path = tempfile.mkstemp(suffix=".ckpt", prefix="weights_only_")
        os.close(fd)
        tmp_ckpt = Path(tmp_path)
        torch.save(minimal, str(tmp_ckpt))
        use_ckpt = tmp_ckpt
        print("DEBUG: saved weights-only temporary checkpoint:", tmp_ckpt)
    else:
        print("INFO: no checkpoint present (will start fresh)")

    batch_override = int(globals().get("BATCH_SIZE", 4))
    num_workers_override = int(globals().get("NUM_WORKERS", 0))
    segment_size_override = int(globals().get("SEGMENT_SIZE", 4096))
    precision = str(globals().get("PRECISION", "16"))

    cli_args = [
        "fit",
        "--data.voice_name",
        VOICE_NAME,
        "--data.csv_path",
        str(DATA_ROOT / "metadata.csv"),
        "--data.audio_dir",
        str(DATA_ROOT / "wavs"),
        "--data.cache_dir",
        str(globals().get("CACHE_DIR", "/content/piper_cache")),
        "--data.batch_size",
        str(batch_override),
        "--data.num_workers",
        str(num_workers_override),
        "--data.shuffle_mode",
        str(globals().get("SHUFFLE", "strong")),
        "--data.config_path",
        str(CONFIG_PATH),
        "--data.espeak_voice",
        globals().get("ESPEAK_VOICE", "pl"),
        "--model.sample_rate",
        str(globals().get("SAMPLE_RATE", 22050)),
        "--model.learning_rate",
        str(globals().get("LEARNING_RATE", 2e-4)),
        "--model.learning_rate_d",
        str(globals().get("LEARNING_RATE_D", 1e-4)),
        "--model.lr_decay",
        str(globals().get("LR_DECAY", 0.999875)),
        "--model.lr_decay_d",
        str(globals().get("LR_DECAY_D", 0.9999)),
        "--model.segment_size",
        str(segment_size_override),
        "--trainer.precision",
        precision,
    ]

    if use_ckpt:
        cli_args += ["--ckpt_path", str(use_ckpt), "--weights_only", "true"]

    if "MAX_EPOCHS" in globals() and MAX_EPOCHS is not None:
        cli_args += ["--trainer.max_epochs", str(MAX_EPOCHS)]

    if torch.cuda.is_available():
        cli_args += ["--trainer.accelerator", "gpu", "--trainer.devices", "1"]

    print("DEBUG: final CLI args:", cli_args)

    launcher = textwrap.dedent(f"""
        import runpy, sys, pathlib, torch
        torch.serialization.add_safe_globals([pathlib.PosixPath])
        sys.argv = ['piper.train'] + {repr(cli_args)}
        runpy.run_module('piper.train', run_name='__main__')
    """)
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(launcher)
        launcher_path = f.name

    print("DEBUG: launcher written to", launcher_path)
    python_exe = sys.executable
    cmd = [python_exe, "-u", launcher_path]
    print("DEBUG: launching training subprocess:", " ".join(cmd))
    ret = subprocess.run(cmd, cwd=str(PIPER_SRC.parent if PIPER_SRC.exists() else os.getcwd()), env=os.environ.copy())
    if ret.returncode != 0:
        raise subprocess.CalledProcessError(ret.returncode, cmd)

if TRAINING_PHASE == "all":
    if "PHASE_ORDER" not in globals():
        raise ValueError("PHASE_ORDER is not defined. Please define it in cell 2.2.")
    for phase in PHASE_ORDER:
        if phase not in PHASE_CONFIGS:
            raise ValueError(f"Unknown training phase in PHASE_ORDER: {phase}")
        print(f"\n=== Starting training phase: {phase} ===")
        for key, value in PHASE_CONFIGS[phase].items():
            globals()[key] = value
        print("MAX_EPOCHS =", MAX_EPOCHS)
        print("BATCH_SIZE =", BATCH_SIZE)
        print("LEARNING_RATE =", LEARNING_RATE)
        print("LEARNING_RATE_D =", LEARNING_RATE_D)
        print("LR_DECAY =", LR_DECAY)
        print("LR_DECAY_D =", LR_DECAY_D)
        run_training_phase(phase)
else:
    run_training_phase(TRAINING_PHASE)


## 4) Export lastet checkpoint to Google Drive

Save the latest checkpoint into your Google Drive for training restart and local export to onnx.
Once the local environment is set up via:

```bash
./bootstrap.sh
source .venv/bin/activate
```

You can export the latest checkpoint from Google Drive to your local environment via:

```bash
python train.py export --checkpoint <checkpoint_in_google_drive> --output <model_name>.onnx
python synth.py --model <model_name>.onnx --play --text "Your text here"
```

In [ ]:
# Copy latest checkpoint to Drive for safekeeping (run in Colab)
from pathlib import Path
import shutil, sys
from datetime import datetime

try:
    DATA_ROOT, DRIVE_ROOT, VOICE_NAME
except NameError:
    raise RuntimeError("Run cell 2 first to set DATA_ROOT, DRIVE_ROOT and VOICE_NAME.")

search_roots = [
    DATA_ROOT / "tts_output",
    DATA_ROOT,
    Path("/content/piper1-gpl/lightning_logs"),
    Path("/content/piper1-gpl"),
    Path("/tmp"),
    Path("/content"),
]

cands = []
for root in search_roots:
    if not root.exists():
        continue
    for p in root.rglob("*.ckpt"):
        cands.append(p.resolve())

if not cands:
    raise FileNotFoundError("No .ckpt files found under expected locations.")

latest = max(cands, key=lambda p: p.stat().st_mtime)
print("Latest checkpoint found:", latest)

# ensure Drive dir exists
dst_dir = Path(DRIVE_ROOT)
dst_dir.mkdir(parents=True, exist_ok=True)

# Human-readable timestamp and single destination filename
timestamp = datetime.utcnow().strftime("%Y-%m-%d_%H-%M-%S")
dst_name = f"{VOICE_NAME}-{timestamp}.ckpt"
dst_path = dst_dir / dst_name
print("Copying to Drive:", dst_path)
shutil.copy2(str(latest), str(dst_path))

# Also write a stable 'latest' copy for quick reference
stable = dst_dir / f"{VOICE_NAME}-latest.ckpt"
try:
    if stable.exists():
        stable.unlink()
    shutil.copy2(str(dst_path), str(stable))
    print("Also wrote stable copy:", stable)
except Exception:
    print("Could not write stable copy (ignored).")

print("Done. Drive copy:", dst_path, f"({dst_path.stat().st_size/1024/1024:.1f} MB)")